## Converting apple raw db to UTC format

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime
from zoneinfo import ZoneInfo

engine = create_engine(f"sqlite:///habits.db")

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    apple_data_raw = pd.read_sql_query(
        text("""SELECT * FROM apple_data_raw"""),
        connection)

# (30 seconds)
datetime_cols = ['startDate', 'endDate', 'creationDate']
for col in datetime_cols:
    apple_data_raw[col] = pd.to_datetime(apple_data_raw[col], format="%Y-%m-%dT%H:%M:%S%z", utc=True)

# Appending the new raw data to my sql lite table (2 min 12 secs)
apple_data_raw.to_sql(name="apple_data_raw", con=engine, if_exists='replace', index=False)

## Converting apple workouts db to UTC format

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime
from zoneinfo import ZoneInfo

engine = create_engine(f"sqlite:///habits.db")

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    apple_workouts = pd.read_sql_query(
        text("""SELECT * FROM apple_workouts"""), 
        connection)


datetime_cols = ['StartDate']
for col in datetime_cols:
    apple_workouts[col] = pd.to_datetime(apple_workouts[col], format="%Y-%m-%d %H:%M:%S%z", utc=True)


# Appending the new raw data to my sql lite table
apple_workouts.to_sql(name="apple_workouts", con=engine, if_exists='replace', index=False)

## Delete Week Period and Month from Apple Workouts db
#### (I'll create them on read in insteads)

## Filter out apple workouts where activity is blank
#### This includes workouts from old devices like Garmin + Triatholon (I Believe)

In [ ]:
delete from apple_workouts 
where activity is NULL 

Add dependency for plotly in jup notebook
##### uv add nbformat

### Fix timezones in habit entries

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime
from zoneinfo import ZoneInfo

engine = create_engine(f"sqlite:///habits.db")

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    habit_entries = pd.read_sql_query(
        text("""SELECT * FROM habit_entries"""), connection)
        #parse_dates=["timestamp"]) # Verifying that conversion works

# Converting timestamp from EST to UTC
#habit_entries["timestamp"] = pd.to_datetime(habit_entries["timestamp"]).dt.tz_localize("US/Eastern").dt.tz_convert("UTC")
habit_entries["timestamp"] = pd.to_datetime(habit_entries["timestamp"], format= "%Y-%m-%d %H:%M:%S", utc = True)

# Appending the new raw data to my sql lite table
habit_entries.to_sql(name="habit_entries", con=engine, if_exists='replace', index=False)

# Need to now recreate the habit entries table with correct keys, nulls, and autoincrements
with engine.connect() as conn:
    # habit_entries table
    conn.execute(text("""CREATE TABLE IF NOT EXISTS habit_entries_new (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        habit_id INTEGER NOT NULL,
                        log_date TEXT NOT NULL,
                        timestamp TEXT NOT NULL,
                        FOREIGN KEY (habit_id) REFERENCES habits(id));"""))

In [ ]:
with engine.begin() as conn:
    # habit_entries table
    conn.execute(text("""INSERT INTO habit_entries_new (id, habit_id, log_date, timestamp)
                        SELECT id, habit_id, log_date, timestamp
                        FROM habit_entries"""))
    
# Delete original habit_entries table from sqlite and rename "habit_entries_new" to "habit_entries" 

### Fix timezones in 10RM completions

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime
from zoneinfo import ZoneInfo

engine = create_engine(f"sqlite:///habits.db")

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    tenrm_completions = pd.read_sql_query(
        text("""SELECT * FROM tenrm_completions"""), connection)
        #parse_dates=["timestamp"]) # Verifying that conversion works

# Converting timestamp from EST to UTC
#tenrm_completions["timestamp"] = pd.to_datetime(tenrm_completions["timestamp"]).dt.tz_localize("US/Eastern").dt.tz_convert("UTC")
tenrm_completions["timestamp"] = pd.to_datetime(tenrm_completions["timestamp"], format= "%Y-%m-%d %H:%M:%S", utc = True)

# Appending the new raw data to my sql lite table
tenrm_completions.to_sql(name="tenrm_completions", con=engine, if_exists='replace', index=False)

In [ ]:
# Need to now recreate the tenrm_completions table with correct keys, nulls, and autoincrements
with engine.connect() as conn:
    # habit_entries table
    conn.execute(text("""CREATE TABLE IF NOT EXISTS tenrm_completions_new (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        plan_id INTEGER NOT NULL,
                        completion_date TEXT NOT NULL,
                        completed INTEGER NOT NULL,
                        notes TEXT,
                        timestamp TEXT NOT NULL,
                        FOREIGN KEY (plan_id) REFERENCES tenrm_plans(id));"""))

In [ ]:
with engine.begin() as conn:
    # tenrm_completions table
    conn.execute(text("""INSERT INTO tenrm_completions_new (id, plan_id, completion_date, completed, notes, timestamp)
                        SELECT id, plan_id, completion_date, completed, notes, timestamp
                        FROM tenrm_completions"""))
    
# Delete original tenrm_completions table from sqlite and rename "tenrm_completions_new" to "tenrm_completions"

### Fix timezones in access logs table

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime
from zoneinfo import ZoneInfo

engine = create_engine(f"sqlite:///habits.db")

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    access_log = pd.read_sql_query(
        text("""SELECT * FROM access_log"""), connection)
        #parse_dates=["timestamp"]) # Verifying that conversion works

# Converting timestamp from EST to UTC
#tenrm_completions["timestamp"] = pd.to_datetime(tenrm_completions["timestamp"]).dt.tz_localize("US/Eastern").dt.tz_convert("UTC")
access_log["timestamp"] = pd.to_datetime(access_log["timestamp"], format= "%Y-%m-%d %H:%M:%S", utc = True)

# Appending the new raw data to my sql lite table
access_log.to_sql(name="access_log", con=engine, if_exists='replace', index=False)

In [ ]:
# Need to now recreate the tenrm_completions table with correct keys, nulls, and autoincrements
with engine.connect() as conn:
    # access_log table
    conn.execute(text("""CREATE TABLE IF NOT EXISTS access_log_new (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        ip TEXT NOT NULL,
                        endpoint TEXT NOT NULL,
                        timestamp TEXT NOT NULL);"""))

In [ ]:
with engine.begin() as conn:
    # tenrm_completions table
    conn.execute(text("""INSERT INTO access_log_new (id, ip, endpoint, timestamp)
                        SELECT id, ip, endpoint, timestamp
                        FROM access_log"""))
    
# Delete original access_log table from sqlite and rename "access_log_new" to "access_log" 